In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 7 Lab: Finite State Machines

## Course: Accelerated HDL for Digital System Design — Week 2, Session 7

---

## Objectives

By the end of this lab, you will:
- Implement a timed FSM (traffic light controller) using the 3-always-block style
- Write a self-checking FSM testbench verifying all state transitions
- Design a button pattern detector FSM
- (Stretch) Compare Moore vs. Mealy implementations side by side

## Prerequisites

- Watched Day 7 pre-class video (~50 min): Moore/Mealy, state diagrams, 3-block template
- `debounce.v` and `hex_to_7seg.v` from previous labs (provided in `starter/`)

## Critical Habit: Draw the State Diagram FIRST

**Do not start coding until you have drawn the state diagram on paper.** This is non-negotiable. The diagram is your design document.

## Deliverables

| # | Exercise | Time | What to Submit |
|---|----------|------|----------------|
| 1 | Traffic Light Controller | 40 min | `traffic_light.v` + `tb_traffic_light.v` — all transitions verified |
| 2 | Button Pattern Detector | 35 min | `pattern_detector.v` + `top_pattern.v` + testbench |
| 3 | Testbench Deep Dive | 25 min | Extended `tb_traffic_light.v` with negative/stability tests |
| 4 | Moore vs. Mealy (stretch) | 20 min | Side-by-side waveform comparison screenshot |

**Primary deliverable:** Traffic light FSM running on board with waveform-verified testbench.

---

## Exercise 1: Traffic Light Controller (40 min)

**SLOs: 7.2, 7.3, 7.5**

### Part A: Implement `traffic_light.v`

Use the starter file — the state encoding and timing parameters are provided. You implement the 3-always-block FSM and the timer.

**States:** GREEN (5s) → YELLOW (1s) → RED (4s) → repeat

**Go Board mapping:** LED1=Red, LED2=Yellow, LED3=Green, LED4=Heartbeat

### Part B: Self-Checking Testbench

Use `tb_traffic_light.v` starter. Implement the `check_state` task and verify:
1. Reset → GREEN
2. GREEN → YELLOW after GREEN_TIME cycles
3. YELLOW → RED after YELLOW_TIME cycles
4. RED → GREEN (full cycle)
5. Mid-state reset → GREEN

### Part C: Hardware Demo

Synthesize with real timer values (remove `SIMULATION` define). Watch the LEDs cycle.

In [ ]:
!make sim TB=tb_traffic_light.v SRCS="traffic_light.v"

In [ ]:
!make PROJECT=traffic_light

In [ ]:
!make prog PROJECT=traffic_light

---

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile hex_to_7seg.v
// hex_to_7seg.v — Provided module (from Day 2)
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b0000001;  4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;  4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;  4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;  4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;  4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;  4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;  4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;  4'hF: o_seg = 7'b0111000;
        endcase
    end
endmodule

In [ ]:
%%writefile tb_traffic_light.v
// =============================================================================
// tb_traffic_light.v — Traffic Light FSM Testbench (Starter)
// Day 7, Exercise 1
// =============================================================================

`timescale 1ns / 1ps
`define SIMULATION

module tb_traffic_light;

    reg  clk, reset;
    wire [2:0] light;

    traffic_light uut (
        .i_clk(clk), .i_reset(reset), .o_light(light)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    // ---- TODO: Implement check_state task ----
    // task check_state;
    //   input [2:0] expected_light;
    //   input [80*8-1:0] label;
    // begin
    //   test_count = test_count + 1;
    //   @(posedge clk); #1;
    //   if (light !== expected_light) begin
    //     fail_count = fail_count + 1;
    //     $display("FAIL [%0t]: %0s — expected %b, got %b",
    //              $time, label, expected_light, light);
    //   end else begin
    //     $display("PASS [%0t]: %0s (light=%b)", $time, label, light);
    //   end
    // end
    // endtask

    task wait_clks;
        input integer n;
        begin repeat (n) @(posedge clk); end
    endtask

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_traffic_light);

        // Reset
        reset = 1;
        wait_clks(3);
        reset = 0;
        @(posedge clk); #1;

        // ---- TODO: Test 1 — Verify initial state is GREEN ----
        // check_state(3'b001, "After reset: GREEN");

        // ---- TODO: Test 2 — Wait GREEN_TIME, verify YELLOW ----
        // GREEN_TIME = 10 in simulation
        // wait_clks(10);
        // check_state(3'b010, "After GREEN timer: YELLOW");

        // ---- TODO: Test 3 — Wait YELLOW_TIME, verify RED ----

        // ---- TODO: Test 4 — Wait RED_TIME, verify GREEN ----

        // ---- TODO: Test 5 — Mid-state reset ----
        // wait_clks(3); reset = 1; wait_clks(2); reset = 0;
        // check_state(3'b001, "After mid-state reset: GREEN");

        // Summary
        $display("");
        $display("========================================");
        $display("Traffic Light: %0d / %0d tests passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile traffic_light.v
// =============================================================================
// traffic_light.v — Timed Traffic Light FSM (Starter)
// Day 7, Exercise 1
// =============================================================================

module traffic_light (
    input  wire       i_clk,
    input  wire       i_reset,
    output reg  [2:0] o_light    // {red, yellow, green}
);

    // ========== State Encoding ==========
    localparam S_GREEN  = 2'b00;
    localparam S_YELLOW = 2'b01;
    localparam S_RED    = 2'b10;

    // ========== Timing Parameters ==========
`ifdef SIMULATION
    localparam GREEN_TIME  = 10;
    localparam YELLOW_TIME = 4;
    localparam RED_TIME    = 8;
`else
    localparam GREEN_TIME  = 25_000_000 * 5;  // 5 seconds
    localparam YELLOW_TIME = 25_000_000 * 1;  // 1 second
    localparam RED_TIME    = 25_000_000 * 4;  // 4 seconds
`endif

    localparam MAX_TIME = GREEN_TIME;  // largest timer value
    reg [$clog2(MAX_TIME)-1:0] r_timer;

    reg [1:0] r_state, r_next_state;

    // ===== Block 1: State Register (Sequential) =====
    always @(posedge i_clk) begin
        if (i_reset)
            r_state <= S_GREEN;
        else
            r_state <= r_next_state;
    end

    // ===== Timer: counts up, resets on state transition =====
    always @(posedge i_clk) begin
        if (i_reset)
            r_timer <= 0;
        else if (r_state != r_next_state)
            r_timer <= 0;                // reset on transition
        else
            r_timer <= r_timer + 1;
    end

    // ===== Block 2: Next-State Logic (Combinational) =====
    always @(*) begin
        r_next_state = r_state;  // DEFAULT: stay

        case (r_state)
            S_GREEN: begin
                // ---- TODO: Transition to S_YELLOW when timer reaches GREEN_TIME-1 ----

            end

            S_YELLOW: begin
                // ---- TODO: Transition to S_RED when timer reaches YELLOW_TIME-1 ----

            end

            S_RED: begin
                // ---- TODO: Transition to S_GREEN when timer reaches RED_TIME-1 ----

            end

            default: r_next_state = S_GREEN;
        endcase
    end

    // ===== Block 3: Output Logic (Moore) =====
    always @(*) begin
        o_light = 3'b000;  // default: all off

        case (r_state)
            // ---- TODO: Set o_light = {red, yellow, green} for each state ----
            // S_GREEN:  o_light = 3'b001;
            // S_YELLOW: o_light = 3'b010;
            // S_RED:    o_light = 3'b100;
            default: o_light = 3'b000;
        endcase
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = traffic_light
SRCS     = hex_to_7seg.v traffic_light.v
TB       = tb_traffic_light.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_traffic_light.v hex_to_7seg.v traffic_light.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 2: Button Pattern Detector (35 min)

**SLOs: 7.2, 7.6**

Detect the sequence: Switch1 → Switch2 → Switch3. Light LED4 for 1 second on detection. Wrong button resets to beginning.

**Draw the state diagram first!** States: WAIT_1, WAIT_2, WAIT_3, DETECTED

Use `starter/pattern_detector.v` and `starter/top_pattern.v`.

**Testbench scenarios (required):**
1. Correct sequence → detection
2. Wrong sequence → no detection
3. Partial correct then wrong: btn1→btn2→btn1 → resets to WAIT_2
4. Double press: btn1→btn1→btn2→btn3 → still detects
5. Reset mid-sequence

---

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Solution)
// Day 5, Exercise 1
// =============================================================================

module debounce #(
    parameter CLKS_TO_STABLE = 250_000
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // 2-FF synchronizer
        r_sync_0 <= i_bouncy;
        r_sync_1 <= r_sync_0;

        // Debounce logic
        if (r_sync_1 != o_clean) begin
            r_count <= r_count + 1;
            if (r_count == CLKS_TO_STABLE - 1) begin
                o_clean <= r_sync_1;
                r_count <= 0;
            end
        end else begin
            r_count <= 0;
        end
    end

endmodule

In [ ]:
%%writefile hex_to_7seg.v
// hex_to_7seg.v — Provided module (from Day 2)
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b0000001;  4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;  4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;  4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;  4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;  4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;  4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;  4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;  4'hF: o_seg = 7'b0111000;
        endcase
    end
endmodule

In [ ]:
%%writefile pattern_detector.v
// =============================================================================
// pattern_detector.v — Button Sequence Detector FSM (Starter)
// Day 7, Exercise 2
// =============================================================================
// Detect: btn1 -> btn2 -> btn3 (in order)
// Wrong button resets. LED lights for ~1 second on detection.

module pattern_detector (
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_btn1,    // debounced, active-high press edges
    input  wire       i_btn2,
    input  wire       i_btn3,
    output reg        o_detected,
    output reg  [1:0] o_progress  // 00=waiting, 01=got 1, 10=got 1+2, 11=done
);

    // ---- TODO: State encoding ----
    localparam S_WAIT_1   = 2'b00;
    localparam S_WAIT_2   = 2'b01;
    localparam S_WAIT_3   = 2'b10;
    localparam S_DETECTED = 2'b11;

    reg [1:0] r_state, r_next_state;

    // Timer for DETECTED state (~1 second display)
`ifdef SIMULATION
    localparam DETECT_TIME = 20;
`else
    localparam DETECT_TIME = 25_000_000;
`endif
    reg [$clog2(DETECT_TIME)-1:0] r_detect_timer;

    // ===== Block 1: State Register =====
    always @(posedge i_clk) begin
        if (i_reset)
            r_state <= S_WAIT_1;
        else
            r_state <= r_next_state;
    end

    // Timer for detected state
    always @(posedge i_clk) begin
        if (i_reset || r_state != S_DETECTED)
            r_detect_timer <= 0;
        else
            r_detect_timer <= r_detect_timer + 1;
    end

    // ===== Block 2: Next-State Logic =====
    always @(*) begin
        r_next_state = r_state;

        case (r_state)
            S_WAIT_1: begin
                // ---- TODO ----
                // btn1 pressed -> S_WAIT_2
                // any other button -> stay (S_WAIT_1)
            end

            S_WAIT_2: begin
                // ---- TODO ----
                // btn2 pressed -> S_WAIT_3
                // btn1 pressed -> S_WAIT_2 (restart valid first press)
                // btn3 pressed -> S_WAIT_1 (wrong sequence)
            end

            S_WAIT_3: begin
                // ---- TODO ----
                // btn3 pressed -> S_DETECTED
                // btn1 pressed -> S_WAIT_2
                // btn2 pressed -> S_WAIT_1 (wrong)
            end

            S_DETECTED: begin
                // ---- TODO ----
                // After DETECT_TIME -> S_WAIT_1
            end

            default: r_next_state = S_WAIT_1;
        endcase
    end

    // ===== Block 3: Output Logic (Moore) =====
    always @(*) begin
        o_detected  = 1'b0;
        o_progress  = 2'b00;

        case (r_state)
            // ---- TODO: Set outputs for each state ----
            // S_WAIT_1:   o_progress = 2'b00;
            // S_WAIT_2:   o_progress = 2'b01;
            // S_WAIT_3:   o_progress = 2'b10;
            // S_DETECTED: o_progress = 2'b11; o_detected = 1'b1;
            default: ;
        endcase
    end

endmodule

In [ ]:
%%writefile tb_pattern_detector.v
// =============================================================================
// tb_pattern_detector.v — Pattern Detector Testbench (Starter)
// Day 7, Exercise 2
// =============================================================================

`timescale 1ns / 1ps
`define SIMULATION

module tb_pattern_detector;

    reg clk, reset;
    reg btn1, btn2, btn3;
    wire detected;
    wire [1:0] progress;

    pattern_detector uut (
        .i_clk(clk), .i_reset(reset),
        .i_btn1(btn1), .i_btn2(btn2), .i_btn3(btn3),
        .o_detected(detected), .o_progress(progress)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    // ---- TODO: Create a press_btn task ----
    // Pulse the appropriate button signal high for one clock cycle

    // ---- TODO: Create a check task ----
    // Compare progress and detected against expected values

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_pattern_detector);
        btn1 = 0; btn2 = 0; btn3 = 0;
        reset = 1; repeat (3) @(posedge clk); reset = 0;

        // ---- TODO: Test 1 — Correct sequence btn1->btn2->btn3 ----

        // ---- TODO: Test 2 — Wrong sequence (btn2 first) ----

        // ---- TODO: Test 3 — Partial correct then btn1 ----

        // ---- TODO: Test 4 — Reset mid-sequence ----

        $display("");
        $display("========================================");
        $display("Pattern Detector: %0d / %0d tests passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile top_pattern.v
// =============================================================================
// top_pattern.v — Go Board Top Level for Pattern Detector (Starter)
// Day 7, Exercise 2
// =============================================================================

module top_pattern (
    input  wire i_clk,
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,  // reset
    output wire o_led1,     // progress bit 1
    output wire o_led2,     // progress bit 0
    output wire o_led3,     // heartbeat
    output wire o_led4      // DETECTED!
);

    // ---- TODO: Debounce all 4 buttons ----
    wire w_sw1_clean, w_sw2_clean, w_sw3_clean, w_reset_clean;

    // debounce #(.CLKS_TO_STABLE(250_000)) db1 (...);
    // debounce #(.CLKS_TO_STABLE(250_000)) db2 (...);
    // debounce #(.CLKS_TO_STABLE(250_000)) db3 (...);
    // debounce #(.CLKS_TO_STABLE(250_000)) db_reset (...);

    // ---- TODO: Edge detect on buttons 1-3 ----
    // reg r_sw1_prev, r_sw2_prev, r_sw3_prev;
    // wire w_btn1 = ~w_sw1_clean & r_sw1_prev;  // falling edge = press
    // wire w_btn2 = ~w_sw2_clean & r_sw2_prev;
    // wire w_btn3 = ~w_sw3_clean & r_sw3_prev;

    // ---- TODO: Instantiate pattern_detector ----
    wire w_detected;
    wire [1:0] w_progress;

    // pattern_detector pd (...);

    // ---- TODO: Map outputs to LEDs ----
    // assign o_led1 = ~w_progress[1];   // active-low LED
    // assign o_led2 = ~w_progress[0];
    // assign o_led4 = ~w_detected;

    // Heartbeat
    reg [23:0] r_hb;
    always @(posedge i_clk) r_hb <= r_hb + 1;
    assign o_led3 = ~r_hb[23];

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = top_pattern
SRCS     = debounce.v hex_to_7seg.v pattern_detector.v top_pattern.v
TB       = tb_pattern_detector.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_pattern_detector.v debounce.v hex_to_7seg.v pattern_detector.v top_pattern.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 3: Testbench Deep Dive (25 min)

**SLO: 7.4**

Extend the traffic light testbench with:
1. **Negative test:** Verify GREEN doesn't transition at GREEN_TIME-1
2. **Stability test:** Sample outputs multiple times during each state
3. **Full cycle test:** Run 3 complete cycles, verify consistent timing
4. **State check:** Use hierarchical access (`uut.r_state`) to verify no illegal states

---

## Exercise 4 (Stretch): Moore vs. Mealy Comparison (20 min)

**SLO: 7.1**

Implement a "10" sequence detector in both Moore and Mealy styles. Show in GTKWave that the Mealy output appears 1 clock cycle earlier.